In [27]:
import os
from azure.identity import DefaultAzureCredential
from azure.mgmt.subscription import SubscriptionClient
from dotenv import set_key

def select_and_save_subscription():
    credential = DefaultAzureCredential()
    client = SubscriptionClient(credential)
    
    print("Fetching available Azure Subscriptions...\n")
    
    try:
        # CRITICAL FIX: Convert the iterator to a list immediately
        # This allows us to use it multiple times (for printing AND selecting)
        subscriptions_iterator = client.subscriptions.list()
        sub_list = list(subscriptions_iterator)
        
        count = len(sub_list)
        
        if count == 0:
            print("No subscriptions found. Check your Azure account permissions.")
            return

        # Print the options
        print("-" * 50)
        for index, sub in enumerate(sub_list):
            print(f"[{index + 1}] {sub.display_name}")
            print(f"    ID: {sub.subscription_id}")
            print("-" * 50)
            
    except Exception as e:
        print(f"An error occurred fetching subscriptions: {e}")
        return

    # User Input Logic
    id_no = input("\nEnter the subscription number you want to set in .env (or press Enter to skip): ")
    
    if id_no.isdigit() and 1 <= int(id_no) <= count:
        selected_sub = sub_list[int(id_no) - 1]
        
        # Define the .env path (creates it in the current working directory if it doesn't exist)
        env_path = os.path.join(os.getcwd(), ".env")
        
        # Use python-dotenv to set the key safely
        set_key(dotenv_path=env_path, key_to_set="Azure_Subscription_ID", value_to_set=selected_sub.subscription_id)
        
        print(f"\n✅ Set Azure_Subscription_ID to '{selected_sub.subscription_id}' in {env_path}")
    elif id_no == "":
        print("\nSkipping .env update.")
    else:
        print("\n⚠️ Invalid selection. Exiting.")

if __name__ == "__main__":
    select_and_save_subscription()

Fetching available Azure Subscriptions...

--------------------------------------------------
[1] cloudlabs17
    ID: 85defe94-8a06-44e4-89ed-23f42f9f6152
--------------------------------------------------

✅ Set Azure_Subscription_ID to '85defe94-8a06-44e4-89ed-23f42f9f6152' in d:\New folder\src\resource_creation\.env


In [28]:
from azure.mgmt.resource import ResourceManagementClient
from dotenv import get_key
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
subscription_id = get_key(".env", "Azure_Subscription_ID")
def select_and_save_resource_group():
    # 1. Authenticate
    credential = DefaultAzureCredential()
    
    # 2. Initialize the Resource Management Client
    client = ResourceManagementClient(credential, subscription_id)
    
    print("Fetching Resource Groups from Azure...\n")
    
    # 3. Retrieve and store the resource groups in a list
    try:
        rg_iterator = client.resource_groups.list()
        rg_list = list(rg_iterator) # Convert iterator to a standard list
    except Exception as e:
        print(f"❌ Failed to fetch resource groups: {e}")
        return

    # Handle case where subscription has no resource groups
    if not rg_list:
        print("No resource groups found in this subscription.")
        return

    # 4. Display the interactive menu
    print("Available Resource Groups:")
    print("-" * 30)
    for index, rg in enumerate(rg_list):
        print(f"[{index + 1}] {rg.name} (Location: {rg.location})")
    print("-" * 30)
    
    # 5. Capture and validate user input
    selected_rg_name = None
    while True:
        try:
            choice = input("\nEnter the number of the Resource Group to select: ")
            choice_index = int(choice) - 1
            
            # Check if the number is within the valid range
            if 0 <= choice_index < len(rg_list):
                selected_rg_name = rg_list[choice_index].name
                break
            else:
                print("⚠️ Invalid number. Please select a number from the list.")
        except ValueError:
            print("⚠️ Invalid input. Please enter a numerical value.")

    print(f"\n✅ You selected: {selected_rg_name}")

    # 6. Write the selection to the .env file
    env_file_path = ".env"
    set_key(dotenv_path=env_file_path, key_to_set="RESOURCE_GROUP_NAME", value_to_set=selected_rg_name)   
    print(f"📁 Successfully wrote 'RESOURCE_GROUP_NAME={selected_rg_name}' to {env_file_path}")

if __name__ == "__main__":
    select_and_save_resource_group()

Fetching Resource Groups from Azure...

Available Resource Groups:
------------------------------
[1] user-qmekphcfmvuc (Location: eastus)
------------------------------

✅ You selected: user-qmekphcfmvuc
📁 Successfully wrote 'RESOURCE_GROUP_NAME=user-qmekphcfmvuc' to .env


In [29]:
import os
from dotenv import load_dotenv, set_key
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import (
    Account,
    Sku,
    AccountProperties,
)


def main():
    env_path = ".env"

    # Load .env
    load_dotenv(env_path, override=True)

    subscription_id = os.getenv("Azure_Subscription_ID")
    resource_group = os.getenv("RESOURCE_GROUP_NAME")

    if not subscription_id or not resource_group:
        print(
            "Azure_Subscription_ID or RESOURCE_GROUP_NAME is missing from the .env file."
        )
        return

    print("Authenticating with Azure...")

    credential = DefaultAzureCredential()

    client = CognitiveServicesManagementClient(
        credential,
        subscription_id,
    )

    print(f"Searching for Foundry resources in '{resource_group}'...\n")

    try:
        accounts = list(
            client.accounts.list_by_resource_group(resource_group)
        )

        foundry_resources = [
            account
            for account in accounts
            if account.kind == "AIServices"
        ]

    except Exception as e:
        print("Unable to retrieve resources.")
        print(e)
        return

    selected_resource_name = None
    selected_resource_endpoint = None

    # ------------------------------------------------------------------
    # CREATE NEW RESOURCE
    # ------------------------------------------------------------------
    if not foundry_resources:

        print("No Microsoft Foundry resource found.")

        new_name = input(
            "Enter a globally unique resource name: "
        )

        location = input(
            "Enter Azure Region (ex: eastus): "
        )

        print("\nCreating resource...")
        print("This may take several minutes.\n")

        account = Account(
            location=location,
            kind="AIServices",
            sku=Sku(name="S0"),
            properties=AccountProperties(
                custom_sub_domain_name=new_name
            ),
        )

        poller = client.accounts.begin_create(
            resource_group_name=resource_group,
            account_name=new_name,
            account=account,
        )

        poller.result()

        created_resource = client.accounts.get(
            resource_group_name=resource_group,
            account_name=new_name,
        )

        selected_resource_name = created_resource.name
        selected_resource_endpoint = (
            created_resource.properties.endpoint
        )

        print("\nResource created successfully.")

    # ------------------------------------------------------------------
    # SELECT EXISTING RESOURCE
    # ------------------------------------------------------------------
    else:

        print("Available Microsoft Foundry Resources:\n")

        for i, resource in enumerate(foundry_resources, start=1):
            print(
                f"{i}. {resource.name}"
            )

        while True:

            try:

                choice = int(
                    input("\nSelect resource number: ")
                )

                if 1 <= choice <= len(foundry_resources):
                    break

                print("Invalid selection.")

            except ValueError:
                print("Please enter a valid number.")

        selected_resource = foundry_resources[choice - 1]

        selected_resource_name = selected_resource.name
        selected_resource_endpoint = (
            selected_resource.properties.endpoint
        )

    # ------------------------------------------------------------------
    # SAVE TO .ENV
    # ------------------------------------------------------------------

    if selected_resource_name:

        set_key(
            env_path,
            "MICROSOFT_FOUNDRY_RESOURCE",
            selected_resource_name,
        )

    if selected_resource_endpoint:

        set_key(
            env_path,
            "MICROSOFT_FOUNDRY_ENDPOINT",
            selected_resource_endpoint,
        )

    print("\n--------------------------------------")
    print("Saved Successfully")
    print("--------------------------------------")
    print(
        f"MICROSOFT_FOUNDRY_RESOURCE = {selected_resource_name}"
    )
    print(
        f"MICROSOFT_FOUNDRY_ENDPOINT = {selected_resource_endpoint}"
    )


if __name__ == "__main__":
    main()

Authenticating with Azure...
Searching for Foundry resources in 'user-qmekphcfmvuc'...

No Microsoft Foundry resource found.

Creating resource...
This may take several minutes.


Resource created successfully.

--------------------------------------
Saved Successfully
--------------------------------------
MICROSOFT_FOUNDRY_RESOURCE = resumeglobalfoundry
MICROSOFT_FOUNDRY_ENDPOINT = https://resumeglobalfoundry.cognitiveservices.azure.com/


In [30]:

from dotenv import load_dotenv


def enable_project_management():
    # 1. Load existing variables from .env
    load_dotenv(override=True)
    subscription_id = os.getenv("Azure_Subscription_ID")
    rg_name = os.getenv("RESOURCE_GROUP_NAME")
    hub_name = os.getenv("MICROSOFT_FOUNDRY_RESOURCE")
    
    if not all([subscription_id, rg_name, hub_name]):
        print("⚠️ Missing environment variables. Make sure your .env file is populated.")
        return

    # 2. Authenticate and Initialize Client
    credential = DefaultAzureCredential()
    client = CognitiveServicesManagementClient(credential, subscription_id)
    
    print(f"Fetching parent Hub '{hub_name}'...")
    
    try:
        # 3. Retrieve the existing Hub configuration
        account = client.accounts.get(resource_group_name=rg_name, account_name=hub_name)
        
        # 4. Enable the project management capability
        account.properties.allow_project_management = True
        
        print("Patching resource to allow child projects. This may take a minute...")
        
        # 5. Push the update back to Azure (begin_create acts as a PUT/upsert)
        poller = client.accounts.begin_create(
            resource_group_name=rg_name,
            account_name=hub_name,
            account=account
        )
        
        poller.result()
        print("\n✅ Hub updated successfully!")
        print("➡️ You can now run your project creation script again for 'tcsglobalpolicy'.")
        
    except Exception as e:
        print(f"\n❌ Failed to update resource: {e}")

if __name__ == "__main__":
    enable_project_management()

Fetching parent Hub 'resumeglobalfoundry'...
Patching resource to allow child projects. This may take a minute...

✅ Hub updated successfully!
➡️ You can now run your project creation script again for 'tcsglobalpolicy'.


In [31]:
load_dotenv(override = True)  # Ensure environment variables are loaded
location = os.getenv("location")
def create_foundry_project(project_name):
    # 1. Authenticate using active session
    credential = DefaultAzureCredential()
    
    # 2. Initialize the Management Client
    client = CognitiveServicesManagementClient(credential, subscription_id)
    
    # 3. Define the project parameters
    project_parameters = {
        "location": location,
        "identity": {
            "type": "SystemAssigned"
        },
        "properties": {} 
    }
    
    print(f"\nProvisioning Foundry project '{project_name}'...")
    
    # 4. Initiate the creation (Asynchronous)
    try:
        poller = client.projects.begin_create(
            resource_group_name=os.getenv("RESOURCE_GROUP_NAME"),  # Ensure this is set in your .env
            account_name=os.getenv("MICROSOFT_FOUNDRY_RESOURCE"),  # Ensure this is set in your .env
            project_name=project_name, # Passed from the argument
            project=project_parameters
        )
        
        # Wait for the creation to finish
        project_result = poller.result()
        print(f"✅ Project created successfully!")
        print(f"Project ID: {project_result.id}")
        set_key(dotenv_path=".env", key_to_set="PROJECT_ID", value_to_set=project_result.id)
    except Exception as e:
        print(f"❌ Failed to create project: {e}")

if __name__ == "__main__":
    print("--- Azure Foundry Project Setup ---")
    # Loop until the user provides a valid, non-empty string
    while True:
        user_project_name = input("Enter a name for your new Foundry project: ").strip()
        
        if user_project_name:
            # Exit loop and run function once a valid name is given
            create_foundry_project(user_project_name)
            break
        else:
            print("⚠️ Project name cannot be empty. Please try again.\n")

--- Azure Foundry Project Setup ---

Provisioning Foundry project 'resumeanalyser'...
✅ Project created successfully!
Project ID: /subscriptions/85defe94-8a06-44e4-89ed-23f42f9f6152/resourceGroups/user-qmekphcfmvuc/providers/Microsoft.CognitiveServices/accounts/resumeglobalfoundry/projects/resumeanalyser


In [32]:
import os
from dotenv import load_dotenv, set_key
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

def deploy_chat_model():
    # 1. Force absolute pathing for Jupyter Notebook environments
    env_path = os.path.abspath(".env")
    
    # override=True forces Jupyter to update cached variables if they changed
    load_dotenv(dotenv_path=env_path, override=True) 
    
    subscription_id = os.getenv("Azure_Subscription_ID")
    rg_name = os.getenv("RESOURCE_GROUP_NAME")
    hub_name = os.getenv("MICROSOFT_FOUNDRY_RESOURCE")
    
    # --- SAFETY CHECK ---
    missing_vars = []
    if not subscription_id: missing_vars.append("Azure_Subscription_ID")
    if not rg_name: missing_vars.append("RESOURCE_GROUP_NAME")
    if not hub_name: missing_vars.append("FOUNDRY_RESOURCE_NAME")
    
    if missing_vars:
        print(f"❌ ERROR: Missing variables in .env file: {', '.join(missing_vars)}")
        print(f"Make sure your .env file is populated and exists at:\n{env_path}")
        return
    # --------------------
    
    client = CognitiveServicesManagementClient(DefaultAzureCredential(), subscription_id)
    deployment_name = "gpt-5.1-deployment"
    
    deployment_params = {
        "properties": {
            "model": {
                "format": "OpenAI",
                "name": "gpt-5.1",
                
            }
        },
        "sku": {"name": "Standard", "capacity": 10} # 10K Tokens per minute
    }

    print(f"Deploying Chat Model '{deployment_name}' to '{hub_name}'...")
    
    try:
        poller = client.deployments.begin_create_or_update(
            resource_group_name=rg_name,
            account_name=hub_name,
            deployment_name=deployment_name,
            deployment=deployment_params
        )
        result = poller.result()
        
        # The endpoint is inherited from the parent Hub
        hub_info = client.accounts.get(rg_name, hub_name)
        endpoint = hub_info.properties.endpoint
        
        set_key(env_path, "CHAT_MODEL_DEPLOYMENT_NAME", result.name)
        set_key(env_path, "AZURE_OPENAI_ENDPOINT", endpoint)
        print(f"✅ Chat Model Deployed! Saved Deployment Name and Endpoint to .env")
        
    except Exception as e:
        print(f"\n❌ Deployment Failed: {e}")

if __name__ == "__main__":
    deploy_chat_model()

Deploying Chat Model 'gpt-5.1-deployment' to 'resumeglobalfoundry'...
✅ Chat Model Deployed! Saved Deployment Name and Endpoint to .env


In [33]:
import os
from dotenv import load_dotenv, set_key


def deploy_embedding_model():
    load_dotenv(override=True)  # Load environment variables from .env file
    subscription_id = os.getenv("Azure_Subscription_ID")
    rg_name = os.getenv("RESOURCE_GROUP_NAME")
    hub_name = os.getenv("MICROSOFT_FOUNDRY_RESOURCE")
    
    client = CognitiveServicesManagementClient(DefaultAzureCredential(), subscription_id)
    deployment_name = "text-embedding-3-small-deployment"
    
    deployment_params = {
        "properties": {
            "model": {
                "format": "OpenAI",
                "name": "text-embedding-3-small",
                "version": "1"
            }
        },
        "sku": {"name": "Standard", "capacity": 50} 
    }

    print(f"Deploying Embedding Model '{deployment_name}'...")
    poller = client.deployments.begin_create_or_update(
        resource_group_name=rg_name,
        account_name=hub_name,
        deployment_name=deployment_name,
        deployment=deployment_params
    )
    result = poller.result()
    
    env_path = os.path.join(os.getcwd(), ".env")
    set_key(env_path, "EMBEDDING_MODEL_DEPLOYMENT_NAME", result.name)
    print(f"✅ Embedding Model Deployed! Saved to .env")

if __name__ == "__main__":
    deploy_embedding_model()

Deploying Embedding Model 'text-embedding-3-small-deployment'...
✅ Embedding Model Deployed! Saved to .env


In [34]:
import os
from dotenv import load_dotenv, set_key
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.core.exceptions import HttpResponseError

def setup_document_intelligence():
    env_path = os.path.abspath(".env")
    load_dotenv(dotenv_path=env_path, override=True)
    
    subscription_id = os.getenv("Azure_Subscription_ID")
    rg_name = os.getenv("RESOURCE_GROUP_NAME")
    
    if not subscription_id or not rg_name:
        print("❌ ERROR: Missing Subscription ID or Resource Group in .env.")
        return

    client = CognitiveServicesManagementClient(DefaultAzureCredential(), subscription_id)
    doc_intel_name = input("Enter a name for Document Intelligence (e.g., resume-doc-intelligence): ").strip()
    
    doc_intel_params = {
        "location": "eastus",
        "kind": "FormRecognizer", # ARM internal name for Document Intelligence
        "sku": {"name": "S0"},
        "identity": {"type": "SystemAssigned"}
    }

    print(f"\nAttempting to provision '{doc_intel_name}'...")
    
    try:
        # 1. Try to create the resource
        poller = client.accounts.begin_create(rg_name, doc_intel_name, doc_intel_params)
        result = poller.result()
        print(f"✅ Successfully created new Document Intelligence service!")
        
    except HttpResponseError as e:
        # 2. If blocked by the lab environment, search for an existing one
        if "AuthorizationFailed" in e.message:
            print(f"⚠️ Creation blocked by lab permissions. Scanning for pre-provisioned services...")
            accounts = list(client.accounts.list_by_resource_group(rg_name))
            form_recognizers = [acc for acc in accounts if acc.kind == "FormRecognizer"]
            
            if not form_recognizers:
                print("❌ No existing Document Intelligence services found. Check your lab instructions.")
                return
                
            result = form_recognizers[0]
            doc_intel_name = result.name
            print(f"✅ Found existing service: '{doc_intel_name}'")
        else:
            print(f"❌ Failed to create service: {e}")
            return

    # 3. Fetch the API Keys and save to .env
    keys = client.accounts.list_keys(rg_name, doc_intel_name)
    
    set_key(env_path, "DOCUMENT_INTELLIGENCE_ENDPOINT", result.properties.endpoint)
    set_key(env_path, "DOCUMENT_INTELLIGENCE_KEY", keys.key1)
    
    print(f"📁 Endpoint and API Key saved to .env!")

if __name__ == "__main__":
    setup_document_intelligence()


Attempting to provision 'resumedi'...
✅ Successfully created new Document Intelligence service!
📁 Endpoint and API Key saved to .env!


In [26]:
import os
from dotenv import load_dotenv, set_key
from azure.identity import DefaultAzureCredential
from azure.mgmt.storage import StorageManagementClient
from azure.core.exceptions import HttpResponseError

def setup_storage_account():
    env_path = os.path.abspath(".env")
    load_dotenv(dotenv_path=env_path, override=True)
    
    subscription_id = os.getenv("Azure_Subscription_ID")
    rg_name = os.getenv("RESOURCE_GROUP_NAME")
    
    client = StorageManagementClient(DefaultAzureCredential(), subscription_id)
    storage_name = input("Enter a globally unique name for Storage (lowercase, numbers only): ").strip()
    
    storage_params = {
        "location": "eastus",
        "sku": {"name": "Standard_LRS"},
        "kind": "StorageV2",
        "properties": {"allowBlobPublicAccess": False}
    }

    print(f"\nAttempting to provision Storage Account '{storage_name}'...")
    
    try:
        # 1. Try to create the resource
        poller = client.storage_accounts.begin_create(rg_name, storage_name, storage_params)
        result = poller.result()
        print(f"✅ Successfully created new Storage Account!")
        
    except HttpResponseError as e:
        # 2. If blocked, search for an existing one
        if "AuthorizationFailed" in e.message:
            print(f"⚠️ Creation blocked by lab permissions. Scanning for pre-provisioned storage...")
            accounts = list(client.storage_accounts.list_by_resource_group(rg_name))
            
            if not accounts:
                print("❌ No existing Storage Accounts found. Check your lab instructions.")
                return
                
            result = accounts[0]
            storage_name = result.name
            print(f"✅ Found existing Storage Account: '{storage_name}'")
        else:
            print(f"❌ Failed to create storage: {e}")
            return

    # 3. Save the connection details to .env
    # Note: Storage uses Connection Strings or URLs instead of traditional endpoints
    storage_url = f"https://{storage_name}.blob.core.windows.net"
    set_key(env_path, "AZURE_STORAGE_ACCOUNT_NAME", storage_name)
    set_key(env_path, "AZURE_STORAGE_URL", storage_url)
    
    print(f"📁 Storage URL and Account Name saved to .env!")

if __name__ == "__main__":
    setup_storage_account()


Attempting to provision Storage Account ''...
❌ Failed to create storage: (UnsupportedResourceOperation) The resource type 'storageAccounts' does not support this operation.
Code: UnsupportedResourceOperation
Message: The resource type 'storageAccounts' does not support this operation.


In [22]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ResourceExistsError

def create_azure_container():
    # 1. Load environment variables
    env_path = os.path.abspath(".env")
    load_dotenv(dotenv_path=env_path, override=True)
    
    storage_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
    storage_url = os.getenv("AZURE_STORAGE_URL")
    
    # Fallback if URL isn't explicitly set in your environment yet
    if storage_name and not storage_url:
        storage_url = f"https://{storage_name}.blob.core.windows.net"
        
    if not storage_url:
        print("❌ ERROR: Missing AZURE_STORAGE_ACCOUNT_NAME or AZURE_STORAGE_URL in .env.")
        return

    # 2. Prompt for the container name
    container_name = input("Enter the name for your new container (e.g., documents): ").strip().lower()
    
    # 3. Authenticate and initialize Blob Service Client
    print(f"\nConnecting to storage account at: {storage_url}...")
    credential = DefaultAzureCredential()
    blob_service_client = BlobServiceClient(account_url=storage_url, credential=credential)
    
    # 4. Create the container
    print(f"Creating container '{container_name}'...")
    try:
        container_client = blob_service_client.create_container(container_name)
        print(f"✅ Successfully created container: '{container_name}'")
        set_key(env_path, "AZURE_STORAGE_CONTAINER_NAME", container_name)
    except ResourceExistsError:
        print(f"ℹ️ Container '{container_name}' already exists.")
    except Exception as e:
        print(f"❌ Failed to create container: {e}")

if __name__ == "__main__":
    create_azure_container()


Connecting to storage account at: https://resumecheckerstorage.blob.core.windows.net...
Creating container 'documents'...
✅ Successfully created container: 'documents'
